# Import Packages, Import data, Clean table

In [0]:
#packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import seaborn as sns


#Import google analytics data
google_analytics = pd.read_csv("clean_GA.csv")

# Change the datatypes
google_analytics['event_date'] = pd.to_datetime(google_analytics['event_date'])
google_analytics['event_ts_utc'] = pd.to_datetime(google_analytics['event_ts_utc'], utc=True)

# drop NAs in the dataset
google_analytics = google_analytics.dropna(subset=['abandoned'])

## Helper Functions

In [0]:
def analyze_avg_time_between(purchvar, category, eventName):
    # --- 0) Clean datatypes ---
    purch = purchvar[['event_ts_utc', 'purchase_segment']].copy()
    upd   = category[['event_ts_utc', 'purchase_segment']].copy()

    purch['event_ts_utc'] = pd.to_datetime(purch['event_ts_utc'], utc=True)
    upd['event_ts_utc']   = pd.to_datetime(upd['event_ts_utc'],   utc=True)
    purch['purchase_segment'] = purch['purchase_segment'].astype(str)
    upd['purchase_segment']   = upd['purchase_segment'].astype(str)

    # --- 1) Merge purchases and precursors within each segment ---
    merged = purch.merge(upd, on='purchase_segment', suffixes=('_purch', f'_{eventName}'))

    # --- 2) Keep only pairs where precursor happened before purchase ---
    merged = merged.loc[merged['event_ts_utc_purch'] > merged[f'event_ts_utc_{eventName}']].copy()

    # --- 3) (Optional) enforce same-year rule ---
    same_year = merged['event_ts_utc_purch'].dt.year == merged[f'event_ts_utc_{eventName}'].dt.year
    merged = merged.loc[same_year]

    # --- 4) Compute time differences in minutes ---
    merged['mins_between'] = (
        (merged['event_ts_utc_purch'] - merged[f'event_ts_utc_{eventName}'])
        .dt.total_seconds() / 60
    )

    # --- 5) Compute average time gap per segment ---
    out = (
        merged.groupby('purchase_segment', as_index=False)
              .agg(avg_mins_between=('mins_between', 'mean'),
                   num_pairs=('mins_between', 'size'))
              .sort_values('avg_mins_between', ascending=False)
    )

    # --- 6) Print diagnostic summary ---
    print(f"{eventName}: {len(out)} segments analyzed, mean gap (min) across segments = {out['avg_mins_between'].mean():.2f}")

    return out

# --- helpers ---
def top_counts(series, top_n=15):
    s = series.value_counts(dropna=False)
    total = s.sum()
    s = s.head(top_n)
    return pd.DataFrame({
        "pattern": s.index.to_list(),
        "count": s.values,
        "percent": (s.values / total * 100) if total else [0.0]*len(s)
    })

def stream_ngrams_count(seqs, n):
    c = Counter()
    for seq in seqs:
        if isinstance(seq, list) and len(seq) >= n:
            # update directly from a generator; no big intermediate list
            c.update(tuple(seq[i:i+n]) for i in range(len(seq)-n+1))
    return c

def top_ngrams_df(seqs, n=2, top_n=15):
    c = stream_ngrams_count(seqs, n)
    total = sum(c.values())
    rows = [(k, v, (v/total*100 if total else 0.0)) for k, v in c.most_common(top_n)]
    return pd.DataFrame(rows, columns=["pattern","count","percent"])

def compare_tables(df_a, df_b, label_a="abandoned", label_b="completed"):
    a = df_a.rename(columns={"count": f"count_{label_a}", "percent": f"percent_{label_a}"})
    b = df_b.rename(columns={"count": f"count_{label_b}",  "percent": f"percent_{label_b}"})
    # Outer join is fine on the already-small “top N” tables, not on the raw keys
    return a.merge(b, on="pattern", how="outer")

def describeEvents(out, summary_table,eventName):
    print(out['avg_mins_between'].describe())

    perc_75 = out['avg_mins_between'].describe()['75%']
    print("75% of purchases occured in less than", math.ceil(perc_75), "minute(s) of this event taking place.")

    perc_80 = out['avg_mins_between'].quantile(0.85)
    print("85% of purchases occured in less than", math.ceil(perc_80), "minute(s) of this event taking place.")

    perc_90 = out['avg_mins_between'].quantile(0.9)
    print("90% of purchases occured in less than", math.ceil(perc_90), "minute(s) of this event taking place.")

    summary_table.loc[summary_table['pattern'] == eventName, '75%'] = perc_75
    summary_table.loc[summary_table['pattern'] == eventName, '85%'] = perc_80

# --- 1) Segment-level status per (customer_id, purchase_segment) ---
# Priority: recovered > intermediate > not applicable (anything else)
def seg_status(s: pd.Series) -> str:
    sl = s.dropna().astype(str).str.lower()
    if (sl == 'recovered').any():
        return 'recovered'
    if (sl == 'intermediate').any():
        return 'intermediate'
    return 'not applicable'

# --- 3) Grouping logic:
# For each customer, assign a group_id so that
#   - each 'recovered' segment is the END of a group
#   - all PRECEDING consecutive 'intermediate' segments join that same group
#   - a recovered "resets" the run
# Implementation: reverse cumsum of recovered flags
def assign_groups(g: pd.DataFrame) -> pd.DataFrame:
    rec_flag = (g['status'] == 'recovered').astype(int)
    # reverse cumsum so rows before a recovered get same id as the recovered
    grp = rec_flag.iloc[::-1].cumsum().iloc[::-1]
    g = g.copy()
    g['group_id'] = grp
    return g

# --- 5) Build grouped sequences at the (customer_id, group_id) level ---
def collapse_runs(events):
    out, prev = [], object()
    for e in events:
        if e != prev:
            out.append(e)
        prev = e
    return out

# Helper function
def analyze_device(GA,columnName, device_label):
    df = GA[GA[columnName] == device_label]

    # --- Group and process ---
    seg = (
        df.groupby(['customer_id', 'purchase_segment'], as_index=False)
          .agg(
              start_ts=('event_ts_utc', 'min'),
              end_ts=('event_ts_utc', 'max'),
              status=('recovered', seg_status)
          )
    )
    seg = seg.sort_values(['customer_id', 'start_ts'])
    seg = seg.groupby('customer_id', group_keys=False).apply(assign_groups)
    seg_kept = seg[seg['group_id'] > 0].copy()

    df_labeled = df.merge(
        seg_kept[['customer_id', 'purchase_segment', 'group_id', 'status']],
        on=['customer_id', 'purchase_segment'],
        how='inner'
    )

    seq_df = (
        df_labeled.sort_values(['customer_id', 'group_id', 'event_ts_utc'])
        .groupby(['customer_id', 'group_id'], as_index=False)
        .agg(
            segments=('purchase_segment', lambda s: list(pd.unique(s))),
            group_status=('status', 'last'),
            start_ts=('event_ts_utc', 'min'),
            end_ts=('event_ts_utc', 'max'),
            num_events=('event_name', 'size'),
            sequence=('event_name', list),
        )
    )
    seq_df['sequence_collapsed'] = seq_df['sequence'].apply(collapse_runs)

    # --- Duration calculation ---
    seq_df['start_ts'] = pd.to_datetime(seq_df['start_ts'], utc=True, errors='coerce')
    seq_df['end_ts']   = pd.to_datetime(seq_df['end_ts'],   utc=True, errors='coerce')
    seq_df['duration'] = seq_df['end_ts'] - seq_df['start_ts']
    seq_df['duration_hours'] = seq_df['duration'].dt.total_seconds() / 3600
    dur = seq_df['duration_hours'].dropna().to_numpy()
    dur = dur[dur >= 0]

    # --- Duration summary ---
    print(f"\n=== Duration summary (hours) for {device_label} ===")
    print(seq_df['duration_hours'].describe())

    # --- ECDF ---
    p99 = np.percentile(dur, 99) if dur.size else 0.0
    xmax = p99 if np.isfinite(p99) and p99 > 0 else (dur.max() if dur.size else 1.0)
    x = np.sort(dur)
    y = np.arange(1, len(x) + 1) / len(x)
    plt.figure()
    plt.step(x, y, where="post")
    plt.xlim(0, 7000)  # fixed x-axis limit
    plt.xlabel("Time to recovery (hours)")
    plt.ylabel("P(recovered by time ≤ t)")
    plt.title(f"Empirical chance of recovery over time for {device_label} users")
    plt.grid(True)
    plt.show()

    # --- Landmark probabilities ---
    landmarks = [1, 6, 12, 24, 48, 72]
    landmark_probs = {f"{h}h": float((dur <= h).mean()) for h in landmarks}
    out = pd.Series(landmark_probs, name=device_label).to_frame().T
    return out

def plot_recovery_summary(recovery_summary, title_suffix):
    # Ensure column order & numeric dtype
    cols = ['1h', '6h', '12h', '24h', '48h', '72h']
    recovery_summary = recovery_summary[cols].astype(float)

    # ---- Multi-line chart ----
    x_hours = [1, 6, 12, 24, 48, 72]
    x_labels = cols

    fig, ax = plt.subplots(figsize=(8, 5))
    for platform, row in recovery_summary.iterrows():
        ax.plot(x_hours, row.values, marker='o', linewidth=2, label=platform)

    ax.set_xticks(x_hours, x_labels)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time horizon")
    ax.set_ylabel("P(recovered by ≤ t)")
    ax.set_title(f"Recovery Probability {title_suffix}")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Platform", ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()

    # ---- Grouped bar chart ----
    fig, ax = plt.subplots(figsize=(9, 5))
    bar_df = recovery_summary.T
    bar_df.plot(kind='bar', ax=ax)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time horizon")
    ax.set_ylabel("P(recovered by ≤ t)")
    ax.set_title(f"Recovery Probability {title_suffix} (Grouped Bars)")
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Q1 & Q2 Preparation

## Q1 -  What behavioral events or sequence of events are the strongest predictors of cart abandonment?

preparation steps to analyze what leads to cart abandonment:

1.  Filter out irrelevant rows to cart abandonment. This is done by removing rows where abandonment is labeled false because no purchase was made and no items were added to the cart. 
2.  Add the event_page_name to the event_name column for the "page_view" and "button_click" events in order to further contexualize and differentiate between different types of page views and button clicks.
3.  flatten the table to create bins of event sequences in each segment. One row will correlate with one segment and contain all the sequence of events that lead to a successful purchase or an abandoned cart.

Notes: As we saw in the EDA, an overwhelming number of event_names were "page_view" and "button_click". I decided to add the event_page_name to the event_name column for these two actions in order to further contexualize and differentiate between different types of page views and button clicks.

In [0]:
# step 1
GASimplified = google_analytics[~((google_analytics['abandoned'] == False) & (google_analytics['False_by_purchase'] == 'no purchase'))]

# step 2
mask = GASimplified['event_name'].isin(['page_view', 'button_click'])
GASimplified.loc[mask, 'event_name'] = (
    GASimplified.loc[mask, 'event_name'].fillna('') + ' - ' +
    GASimplified.loc[mask, 'event_page_name'].fillna('')
)

# step 3
GASequenceMining = GASimplified.sort_values(['purchase_segment', 'event_ts_utc'])

# Build the sequence of events per segment
seq_dfP1 = (
    GASequenceMining
    .groupby('purchase_segment')
    .agg(
        customer_id=('customer_id', 'first'),
        abandoned=('abandoned', 'first'),
        sequence=('event_name', list),
        start_ts=('event_ts_utc', 'min'),
        end_ts=('event_ts_utc', 'max'),
        num_events=('event_name', 'size')
    )
    .reset_index()
)

# (Optional) Collapse consecutive duplicates within each sequence (reduces noise like repeated page_view,page_view)
def collapse_runs(events):
    out = []
    prev = object()
    for e in events:
        if e != prev:
            out.append(e)
        prev = e
    return out

seq_dfP1['sequence_collapsed'] = seq_dfP1['sequence'].apply(collapse_runs)

## Q2 - What behaviors or conditions lead to a customer returning to complete a previously abandoned cart?

Steps to preparing for modeling what behaviors or conditions lead to cart recovery

1.  Figure out the current rate of cart recovery. 
2.  Filter only for rows that are labeled intermediate or recovery in the recovered column. 'recovery' indicates the segment where an abandoned cart was checked out, and 'intermediate' indicates the segments between an abandoned and recovered cart 
3.  flatten the table to create bins of event sequences in each segment. One row will correlate each recovery and it's preceding intermediate rows. 


In [0]:
# part 1
GAFlat = (google_analytics.groupby('purchase_segment', as_index=False).last())
num_abandoned = (GAFlat['abandoned'] == True).sum()
num_recovered = (GAFlat['recovered'] == 'recovered').sum()

ratio = num_recovered / num_abandoned
pct = ratio * 100

print(f"{pct:.2f}% (recovered of abandoned carts)")

# part 2
GARecovered = google_analytics[((google_analytics['recovered'] == 'recovered') | (google_analytics['recovered'] == 'intermediate'))]

mask = GARecovered['event_name'].isin(['page_view', 'button_click'])
GARecovered.loc[mask, 'event_name'] = (
    GARecovered.loc[mask, 'event_name'].fillna('') + ' - ' +
    GARecovered.loc[mask, 'event_page_name'].fillna('')
)


# part 3
import pandas as pd

GA = GARecovered.copy()

seg = (
    GA.groupby(['customer_id', 'purchase_segment'], as_index=False)
      .agg(
          start_ts=('event_ts_utc', 'min'),
          end_ts=('event_ts_utc', 'max'),
          status=('recovered', seg_status)
      )
)

# --- 2) Order segments within each customer chronologically ---
seg = seg.sort_values(['customer_id', 'start_ts'])

seg = seg.groupby('customer_id', group_keys=False).apply(assign_groups)

seg_kept = seg[seg['group_id'] > 0].copy()

GA_labeled = GA.merge(
    seg_kept[['customer_id', 'purchase_segment', 'group_id', 'status']],
    on=['customer_id', 'purchase_segment'],
    how='inner'  # only events that belong to a recovered-anchored group
)

seq_dfP2 = (
    GA_labeled
      .sort_values(['customer_id', 'group_id', 'event_ts_utc'])
      .groupby(['customer_id', 'group_id'], as_index=False)
      .agg(
          segments=('purchase_segment', lambda s: list(pd.unique(s))),
          group_status=('status', 'last'),           # will be 'recovered'
          start_ts=('event_ts_utc', 'min'),
          end_ts=('event_ts_utc', 'max'),
          num_events=('event_name', 'size'),
          sequence=('event_name', list),
      )
)

seq_dfP2['sequence_collapsed'] = seq_dfP2['sequence'].apply(collapse_runs)


# Q1 and Q2 Modeling

# Q1 -  What behavioral events or sequence of events are the strongest predictors of cart abandonment?

Steps to assess what leads to cart abandonment:

1.  Perform sequence mining on to discover the top first events, last events, bigrams, trigrams, and full sequences that lead to cart abandonment. 
2.  Extract and Analyze the top 10 events or sequence of events that lead to the highest percentage of cart abandonment and append them to a single table for analysis.
3.  Use the results of the time-distance-from-purchase analysis to inform when Swire should reach out to customers after each event is clicked to remind them they have items in their cart. 


## Part 1

In [0]:
# Part 1
# --- split once with boolean masks (no .copy() unless you need it) ---
mask_abd = seq_dfP1["abandoned"].astype(bool)
seqs_abd = seq_dfP1.loc[mask_abd, "sequence"]
seqs_cmp = seq_dfP1.loc[~mask_abd, "sequence"]

# 1) first/last events (vectorized)
first_abd  = top_counts(seqs_abd.str[0], top_n=10)
first_cmp  = top_counts(seqs_cmp.str[0], top_n=10)
last_abd   = top_counts(seqs_abd.str[-1], top_n=10)
last_cmp   = top_counts(seqs_cmp.str[-1], top_n=10)

first_events = compare_tables(first_abd, first_cmp)
last_events  = compare_tables(last_abd, last_cmp)

# 2) full sequences (exact)
# Turn lists -> tuples with a vectorized-ish map once, then value_counts
full_abd = top_counts(seqs_abd.map(lambda s: tuple(s) if isinstance(s, list) else ()), top_n=10)
full_cmp = top_counts(seqs_cmp.map(lambda s: tuple(s) if isinstance(s, list) else ()), top_n=10)
full_sequences = compare_tables(full_abd, full_cmp)

# 3) bigrams / trigrams (streamed; no giant intermediates)
bigrams_abd   = top_ngrams_df(seqs_abd, n=2, top_n=10)
bigrams_cmp   = top_ngrams_df(seqs_cmp, n=2, top_n=10)
trigrams_abd  = top_ngrams_df(seqs_abd, n=3, top_n=10)
trigrams_cmp  = top_ngrams_df(seqs_cmp, n=3, top_n=10)

bigrams  = compare_tables(bigrams_abd,  bigrams_cmp)
trigrams = compare_tables(trigrams_abd, trigrams_cmp)

# Example: show heads for a quick peek (comment out if running in a non-notebook environment)
print("\nTop FIRST events (abandoned vs completed):")
display(first_events.head(15))

print("\nTop LAST events (abandoned vs completed):")
display(last_events.head(15))

print("\nTop FULL sequences (abandoned vs completed):")
display(full_sequences.head(10))

print("\nTop BIGRAMS (abandoned vs completed):")
display(bigrams.head(15))

print("\nTop TRIGRAMS (abandoned vs completed):")
display(trigrams.head(15))

# Part 2

In [0]:
# Create summary_table from the top LAST events output
summary_table = last_events.loc[
    last_events['percent_abandoned'].notna(), 
    ['pattern', 'percent_abandoned']
].copy()

# Add blank (empty) columns for 75% and 85%
summary_table['75%'] = ''
summary_table['85%'] = ''
summary_table.head(n=10)

Results: 
1. The top first events aren't very helpful as it indicates session_starts, dahsboards visits, and purchase success pages from previous purchases. Not much insight can be gleaned from this
2. The top last events are where the most information can be collected. As we can see above, 70% of the instances of cart abandonment we identified above can be explained by these features. Most notably, update_cart. 
3. The full sequences, bigrams, and trigams are not very helpful in predicting cart abandonment. The full sequences are too unique as there isn't a single duplicate pattern identified. The bigrams and trigrams are mostly standard website clicks and don't represent enough of the abandoned dataset.

## Step 3

Analyze the results for each event and append the results to our summary table.

In [0]:
# Part 5
# create purchases table
purchases = GASimplified.loc[
    (GASimplified['event_name'] == 'purchase') &
    (GASimplified['abandoned'] != True)&
    (GASimplified['recovered'] != 'recovered'),
    ['event_name', 'event_ts_utc', 'purchase_segment']
]

# update_cart
update_carts = GASimplified.loc[GASimplified['event_name'] == 'update_cart', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,update_carts,'update_cart')
describeEvents(out,summary_table,'update_cart')

# remove_from_cart
remove_from_cart = GASimplified.loc[GASimplified['event_name'] == 'remove_from_cart', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,remove_from_cart,'remove_from_cart')
describeEvents(out,summary_table,'remove_from_cart')

# Analyze view_item_list
view_item_list = GASimplified.loc[GASimplified['event_name'] == 'view_item_list', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,view_item_list,'view_item_list')
describeEvents(out,summary_table,'view_item_list')

# page_view - Unknown Page
unknown = GASimplified.loc[GASimplified['event_name'] == 'page_view - Unknown Page', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,unknown,'page_view - Unknown Page')
describeEvents(out,summary_table,'page_view - Unknown Page')

# page_view - Mycoke Dashboard
myCokeDash = GASimplified.loc[GASimplified['event_name'] == 'page_view - Mycoke Dashboard', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,myCokeDash,'page_view - Mycoke Dashboard')
describeEvents(out,summary_table,'page_view - Mycoke Dashboard')

# Analyze user_engagement
user_engagement = GASimplified.loc[GASimplified['event_name'] == 'user_engagement', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,user_engagement,'user_engagement')
describeEvents(out,summary_table,'user_engagement')

# button_click - Mycoke Orders - Cart
ordersCart = GASimplified.loc[GASimplified['event_name'] == 'button_click - Mycoke Orders - Cart', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,ordersCart,'button_click - Mycoke Orders - Cart')
describeEvents(out,summary_table,'button_click - Mycoke Orders - Cart')

# page_view - Mycoke Orders
orders = GASimplified.loc[GASimplified['event_name'] == 'page_view - Mycoke Orders', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,orders,'page_view - Mycoke Orders')
describeEvents(out,summary_table,'page_view - Mycoke Orders')

# button_click - Mycoke Orders - Checkout: Review Order
revieworder = GASimplified.loc[GASimplified['event_name'] == 'button_click - Mycoke Orders - Checkout: Review Order', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,revieworder,'button_click - Mycoke Orders - Checkout: Review Order')
describeEvents(out,summary_table,'button_click - Mycoke Orders - Checkout: Review Order')

# Analyze proceed_to_checkout
proceed_to_checkout = GASimplified.loc[GASimplified['event_name'] == 'proceed_to_checkout', ['event_name', 'event_ts_utc', 'purchase_segment']]
out = analyze_avg_time_between(purchases,proceed_to_checkout,'proceed_to_checkout')
describeEvents(out,summary_table,'proceed_to_checkout')

# Part 6
summary_table[['75%', '85%']] = (summary_table[['75%', '85%']].astype(float) / 60).round(0)
display(summary_table.head(n=10))
total75 = summary_table.loc[summary_table['75%'] < 24, 'percent_abandoned'].sum()
print("Please refer to the chart above to see which event_names correlate with a 75% chance of purcase within one day of happening. We can potentially reduce cart abandonment by",round(total75,0),"percent if we reach out to customers who enter any of these events and dont purchase within a day.")
total85 = summary_table.loc[summary_table['85%'] < 24, 'percent_abandoned'].sum()
print("Please refer to the chart above to see which event_names correlate with a 85% chance of purcase within one day of happening. We can potentially reduce cart abandonment by",round(total85,0),"percent if we reach out to customers who enter any of these events and dont purchase within a day.")


Events that on average have a 75% of a purchase taking place within one day of being executed. These are the last events in 70% of abandoned cart segments:
1. update_cart
2. remove_from_cart
3. view_item_list
4. page_view - Unknown Page
5. page_view - Mycoke Dashboard	
6. button_click - Mycoke Orders - Cart
7. page_view - Mycoke Orders
8. button_click - Mycoke Orders - Checkout: Review Order
9. proceed_to_checkout

Events that on average have a 85% of a purchase taking place within one day of being executed. These are the last events in 46% of abandoned cart segments:
1. update_cart
2. remove_from_cart
3. button_click - Mycoke Orders - Cart
4. button_click - Mycoke Orders - Checkout: Review Order
5. proceed_to_checkout

# Q2 - What behaviors or conditions lead to a customer returning to complete a previously abandoned cart?

Steps to model what leads to cart recovery:

1.  Analyze event names for sequence mining to understand if certain events or chain of events influence recovery
2.  Perform analysis to unlock the length of time it takes for all recoveries to take place
3.  Perform analysis on each device type to understand the length of time recovery takes by device type.

## Step 1

In [0]:
# --- focus ONLY on recovered segments ---
if 'group_status' in seq_dfP2.columns:
    mask_rec = seq_dfP2['group_status'].astype(str).str.lower().eq('recovered')
elif 'recovered' in seq_dfP2.columns:
    mask_rec = seq_dfP2['recovered'].astype(str).str.lower().eq('recovered')
else:
    raise KeyError("seq_df needs a 'group_status' or 'recovered' column to identify recovered segments.")

seqs_rec = seq_dfP2.loc[mask_rec, 'sequence']

# 1) first / last events (vectorized)
first_rec = top_counts(seqs_rec.str[0], top_n=15)
last_rec  = top_counts(seqs_rec.str[-1], top_n=15)

# 2) full sequences (exact)
full_rec = top_counts(
    seqs_rec.map(lambda s: tuple(s) if isinstance(s, list) else ()),
    top_n=10
)

# 3) bigrams / trigrams (streamed)
bigrams_rec  = top_ngrams_df(seqs_rec, n=2, top_n=15)
trigrams_rec = top_ngrams_df(seqs_rec, n=3, top_n=15)

# --- quick peek ---
print("\nTop FIRST events (recovered only):")
display(first_rec.head(15))

print("\nTop LAST events (recovered only):")
display(last_rec.head(15))

print("\nTop FULL sequences (recovered only):")
display(full_rec.head(10))

print("\nTop BIGRAMS (recovered only):")
display(bigrams_rec.head(15))

print("\nTop TRIGRAMS (recovered only):")
display(trigrams_rec.head(15))

As we can see from the sequence mining results above, there isn't much insight we can gather. the results are similar to the abandonment results, except the last events are less insightful since 92% are purchase (which makes sense). We should focus on understanding time as a factor for cart recovery

## Step 2

### Step 2.1 Analzye all time-based results for cart recovery

In [0]:
# Ensure timestamps are datetime64[ns, UTC]
seq_dfP2['start_ts'] = pd.to_datetime(seq_dfP2['start_ts'], utc=True, errors='coerce')
seq_dfP2['end_ts']   = pd.to_datetime(seq_dfP2['end_ts'],   utc=True, errors='coerce')

# 1) Compute duration as a timedelta
seq_dfP2['duration'] = seq_dfP2['end_ts'] - seq_dfP2['start_ts']

# 2) Extract in various units for convenience
seq_dfP2['duration_hours'] = seq_dfP2['duration'].dt.total_seconds() / 3600
seq_dfP2['duration_days']  = seq_dfP2['duration'].dt.total_seconds() / 86400
seq_dfP2['duration_minutes'] = seq_dfP2['duration'].dt.total_seconds() / 60

# 3) Quick descriptive statistics
print("\n=== Duration summary (hours) ===")
print(seq_dfP2['duration_hours'].describe())

# 4) Optional: aggregate by customer or group_status
summary = (
    seq_dfP2.groupby('group_status')['duration_hours']
    .describe(percentiles=[.25, .5, .75])
    .round(2)
)
print("\n=== Duration by group_status ===")
print(summary)

# 5) Optional: identify longest/shortest recovered sequences
longest = seq_dfP2.nlargest(5, 'duration_hours')[['customer_id','group_id','duration_hours','num_events']]
shortest = seq_dfP2.nsmallest(5, 'duration_hours')[['customer_id','group_id','duration_hours','num_events']]

print("\nLongest recovered sequences:")
display(longest)

print("\nShortest recovered sequences:")
display(shortest)

### Step 2.2 Plot histograms with 90th percentile and first bin identification

In [0]:
# Ensure duration_hours column exists
seq_dfP2['start_ts'] = pd.to_datetime(seq_dfP2['start_ts'], utc=True, errors='coerce')
seq_dfP2['end_ts']   = pd.to_datetime(seq_dfP2['end_ts'], utc=True, errors='coerce')
seq_dfP2['duration_hours'] = (seq_dfP2['end_ts'] - seq_dfP2['start_ts']).dt.total_seconds() / 3600

# Filter valid durations
durations = seq_dfP2.loc[seq_dfP2['duration_hours'] > 0, 'duration_hours']

# Calculate 90th percentile
p90 = durations.quantile(0.90)

# --- Histogram ---
plt.figure(figsize=(8, 4))
sns.histplot(durations, bins=30, kde=True, color='skyblue')

# Add the 90th percentile line
plt.axvline(p90, color='red', linestyle='--', linewidth=2, label=f"90th percentile = {p90:.1f} hrs")

# Add title and labels
plt.title("Distribution of Recovery Durations (Hours)", fontsize=13, weight='bold')
plt.xlabel("Duration (hours)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

#------------------------------------------------------------------------------------------------------------------

# Assuming durations is already defined
durations = seq_dfP2.loc[seq_dfP2['duration_hours'] > 0, 'duration_hours']

# --- Recreate histogram to get bin edges and counts ---
counts, bin_edges = np.histogram(durations, bins=30)

# --- 1) Extract first bin stats ---
first_bin_count = int(counts[0])
first_bin_start = bin_edges[0]
first_bin_end   = bin_edges[1]
first_bin_range = (first_bin_start, first_bin_end)

# --- 2) Compute percentage ---
total_obs = int(counts.sum())
first_bin_pct = (first_bin_count / total_obs * 100) if total_obs else 0.0

# --- 3) Print results ---
print(f"Total observations: {total_obs:,}")
print(f"Number of observations in first bin: {first_bin_count:,} "
      f"({first_bin_pct:.2f}% of total)")
print(f"Duration range of first bin: {first_bin_range[0]:.2f} to {first_bin_range[1]:.2f} hours")

# --- 4) Optional visualization ---
plt.figure(figsize=(8,4))
sns.histplot(durations, bins=30, kde=True, color='skyblue')
plt.axvline(first_bin_end, color='green', linestyle='--', linewidth=2,
            label=(f"End of 1st bin = {first_bin_end:.1f} hrs\n"
                   f"{first_bin_pct:.1f}% of data"))
plt.title("Histogram of Recovery Durations with First Bin Boundary", fontsize=13, weight='bold')
plt.xlabel("Duration (hours)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

### Step 2.3 plot chance of recovery over time for all customers

In [0]:
# --- Assumes seq_df already has duration computed as in your snippet ---
dur = seq_dfP2['duration_hours'].dropna().to_numpy()
dur = dur[dur >= 0]  # guard against negatives

# Percentile cap for readability
p99 = np.percentile(dur, 99) if dur.size else 0.0
xmax = p99 if np.isfinite(p99) and p99 > 0 else (dur.max() if dur.size else 1.0)

# ===== 1) ECDF: P(recovered within ≤ t) =====
x = np.sort(dur)
y = np.arange(1, len(x) + 1) / len(x)

plt.figure()
plt.step(x, y, where="post")
plt.xlim(0, xmax)
plt.xlabel("Time to recovery (hours)")
plt.ylabel("P(recovered by time ≤ t)")
plt.title("Empirical chance of recovery over time")
plt.grid(True)
plt.show()

# ===== 3) Quick landmarks =====
landmarks = [1, 6, 12, 24, 48, 72]
landmark_probs = {f"{h}h": float((dur <= h).mean()) for h in landmarks}
pd.Series(landmark_probs, name="P(recovered by ≤ t)").to_frame()

We can see very clearly that there is a steep chance of prolongued abandonment over time. Summarized results below:
    50% of carts are recovered in 2 hours.
    65% of carts are recovered in 3 days.
    75% of carts are recovered in 168 hours, or 7 days.
    80% of carts are recovered in 235 hours, or 10 days.
    90% of carts are recovered in 674 hours, or 28 days. 
    
It appears that the steep drop off in recovery over time shows that abandonment is intentional. Customers either return to their carts, or they move on and wait until they need inventory. 

## Step 3 Rate of recovery by device type

### Step 3.1 Rate of recovery by device_category

In [0]:
#Test each device category

desktop_tbl = analyze_device(GARecovered,'device_category', "Desktop")
mobile_tbl  = analyze_device(GARecovered,'device_category', "Mobile")
tablet_tbl  = analyze_device(GARecovered,'device_category', "Tablet")

# combine the results into a single table
recovery_summary = pd.concat([desktop_tbl, mobile_tbl, tablet_tbl])
print("\n=== Combined Recovery Probability Table ===")
display(recovery_summary)

plot_recovery_summary(recovery_summary,'device_category')

As we can see from the summarized table above desktop recover the slowest and tablet users recover the fastest. However, after the 3 day mark, we don't see much of a difference between the 3 devices. 

One thing to note, is that if a tablet user has yet to recover by 6 hours, they won't for 3 days. Whereas desktop and mobile users increase recovery slowly over time.

### Step 3.2 Rate of recovery by device_mobile_brand_name

In [0]:
#Test each device category

google = analyze_device(GARecovered,'device_mobile_brand_name', "Google")
apple  = analyze_device(GARecovered,'device_mobile_brand_name', "Apple")
Microsoft  = analyze_device(GARecovered,'device_mobile_brand_name', "Microsoft")
Samsung  = analyze_device(GARecovered,'device_mobile_brand_name', "Samsung")
Mozilla  = analyze_device(GARecovered,'device_mobile_brand_name', "Mozilla")
Other  = analyze_device(GARecovered,'device_mobile_brand_name', "Other")
Samsung  = analyze_device(GARecovered,'device_mobile_brand_name', "Samsung")


# combine the results into a single table
recovery_summary = pd.concat([google, apple, Microsoft, Samsung, Mozilla, Other, Samsung])
print("\n=== Combined Recovery Probability Table ===")
display(recovery_summary)

plot_recovery_summary(recovery_summary,'device_mobile_brand_name')

### Step 3.3 Rate of recovery by device_operating_system

In [0]:
#Test each device category

windows = analyze_device(GARecovered,'device_operating_system', "Windows")
iOS  = analyze_device(GARecovered,'device_operating_system', "iOS")
Macintosh  = analyze_device(GARecovered,'device_operating_system', "Macintosh")
Android  = analyze_device(GARecovered,'device_operating_system', "Android")
Chrome  = analyze_device(GARecovered,'device_operating_system', "Chrome OS")


# combine the results into a single table
recovery_summary = pd.concat([windows, iOS, Macintosh, Android, Chrome])
print("\n=== Combined Recovery Probability Table ===")
display(recovery_summary)

plot_recovery_summary(recovery_summary,'device_operating_system')